In [ ]:
%run ./nb_ingest_silver_cet_carga_descarga

In [1]:
%run ./nb_utils_api_acto_gestao

StatementMeta(, 1d33bb7d-58f8-4b3f-8396-442986bdc8ef, 6, Finished, Available, Finished)

In [2]:
import pandas as pd
from unidecode import unidecode
import numpy as np


df_solicitacoes_origem = pd.read_parquet("/lakehouse/default/Files/acto_cet/silver_cet_carga_descarga_solicitacoes.parquet")
df_etapas_origem = pd.read_parquet("/lakehouse/default/Files/acto_cet/silver_cet_carga_descarga_etapas.parquet")

def converter_horarios(df_solicitacoes, coluna: str) -> pd.DataFrame:
    s = df_solicitacoes[coluna].astype(str).str.strip()
    mask_hora = s.str.contains(":", na=False)
    minutos = pd.to_numeric(s.where(~mask_hora), errors="coerce")

    s = s.where(~s.str.match(r"^\d{1,2}:\d{2}$"), s + ":00")

    horas_from_str = pd.to_datetime(
        s.where(s.str.contains(":", na=False)),
        format="%H:%M:%S",
        errors="coerce"
    )
    horas_from_min = pd.to_datetime(minutos, unit="m", origin="unix", errors="coerce")
    df_solicitacoes[coluna] = (
        horas_from_str.fillna(horas_from_min)
        .dt.strftime("%H:%M:%S")
    )
    return df_solicitacoes


def tratar_nome_colunas(df_solicitacoes) -> pd.DataFrame:

    df_solicitacoes.columns = (
        df_solicitacoes.columns.str.split("|")
        .str[0]  # Get first part before pipe
        .str.strip()  # Remove leading/trailing whitespace
        .map(unidecode)  # Remove accents
        .str.replace(
            r"[:\s]+", "_", regex=True
        )  # Replace colons and spaces with underscore
        .str.replace(r"_+", "_", regex=True)  # Replace multiple underscores with single
        .str.strip("_")  # Remove leading/trailing underscores
        .str.lower()  # Convert to lowercase
    )

    rename_map = {"no_solicitacao": "os", "data_criacao": "data_solicitacao"}
    df_solicitacoes = df_solicitacoes.rename(columns=rename_map)

    return df_solicitacoes


def tratar_datas(df_solicitacoes) -> pd.DataFrame:

    cols = ["data_solicitacao", "data_finalizacao"]

    df_solicitacoes[cols] = (
        df_solicitacoes[cols]
        .apply(lambda s: pd.to_datetime(s, utc=True, errors="coerce", format="ISO8601"))
        .apply(lambda s: s.dt.strftime("%Y-%m-%d %H:%M:%S"))
    )

    df_solicitacoes['dia_da_semana_num'] = pd.to_datetime(df_solicitacoes['data_finalizacao']).dt.weekday.astype(int)
    mapa = {
        0: "segunda-feira",
        1: "terça-feira",
        2: "quarta-feira",
        3: "quinta-feira",
        4: "sexta-feira",
        5: "sábado",
        6: "domingo"
    }
    df_solicitacoes['dia_da_semana_txt'] = df_solicitacoes['dia_da_semana_num'].map(mapa)

    return df_solicitacoes


def adicionar_periodo_dia(df_solicitacoes: pd.DataFrame, coluna: str):

    def classificar_periodo(h):
        if pd.isna(h):
            return None  # ou np.nan, ou "desconhecido"
        
        hora = h.hour
        
        if 0 <= hora < 6:
            return "madrugada"
        elif 6 <= hora < 12:
            return "manhã"
        elif 12 <= hora < 18:
            return "tarde"
        else:
            return "noite"

    df_solicitacoes["horario_dt"] = pd.to_datetime(
        df_solicitacoes[coluna],
        format="%H:%M:%S",
        errors="coerce"   # garante NaT se vier lixo
    )

    str_coluna_nova = "periodo_" + coluna 

    df_solicitacoes[str_coluna_nova] = df_solicitacoes["horario_dt"].apply(classificar_periodo)
    df_solicitacoes = df_solicitacoes.drop(columns=["horario_dt"])

    return df_solicitacoes


def tratar_bairro_carga(df_solicitacoes):
    
    df_solicitacoes["bairro_carga"] = df_solicitacoes["bairro_carga"].str.title()

    replace_bairros_map = {
        "Ponta Da Praia": "Ponta da Praia",
        "Chico De Paula": "Chico de Paula",
    }
    df_solicitacoes["bairro_carga"] = df_solicitacoes["bairro_carga"].replace(
        replace_bairros_map
    )

    df_solicitacoes['bairro_carga'] = (
        df_solicitacoes['bairro_carga']
        .replace("", np.nan)
        .fillna("Indisponível")
    )

    return df_solicitacoes




def main(df_solicitacoes_origem) -> pd.DataFrame:

    df_solicitacoes = df_solicitacoes_origem.copy()

    df_solicitacoes = tratar_nome_colunas(df_solicitacoes_origem)
    df_solicitacoes = tratar_datas(df_solicitacoes)

    df_solicitacoes = converter_horarios(df_solicitacoes, "horario_de_inicio_definido")
    df_solicitacoes = converter_horarios(df_solicitacoes, "horario_de_termino_definido")

    df_solicitacoes = adicionar_periodo_dia(df_solicitacoes, "horario_de_inicio_definido")
    df_solicitacoes = adicionar_periodo_dia(df_solicitacoes, "horario_de_termino_definido")

    df_solicitacoes = tratar_bairro_carga(df_solicitacoes)

    df_solicitacoes = remover_registros_teste(df_solicitacoes)

    df_solicitacoes = df_solicitacoes.drop(
        columns=[
            "quantidade_de_veiculos_autorizados",
            "bairro_descarga",
            "horario_de_inicio",
            "horario_de_termino",
        ]
    )
    return df_solicitacoes

StatementMeta(, 1d33bb7d-58f8-4b3f-8396-442986bdc8ef, 7, Finished, Available, Finished)

In [3]:
df_solicitacoes = main(df_solicitacoes_origem)

StatementMeta(, 1d33bb7d-58f8-4b3f-8396-442986bdc8ef, 8, Finished, Available, Finished)

In [4]:
len(df_solicitacoes)

StatementMeta(, 1d33bb7d-58f8-4b3f-8396-442986bdc8ef, 9, Finished, Available, Finished)

1046

In [5]:
sdf_solicitacoes = spark.createDataFrame(df_solicitacoes)
(
    sdf_solicitacoes
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_cet_carga_descarga")
)

StatementMeta(, 1d33bb7d-58f8-4b3f-8396-442986bdc8ef, 10, Cancelled, Cancelling, Cancelling)